In [1]:
import pandas as pd
import sqlalchemy
import pyodbc

pd.__version__, sqlalchemy.__version__, pyodbc.version


('2.3.3', '2.0.45', '5.3.0')

In [5]:
from sqlalchemy import create_engine

server = "localhost,1433"
database = "TransportAnalysis"
username = "sa"
password = "StrongPass!123"
driver = "ODBC Driver 18 for SQL Server"

connection_string = (
    f"mssql+pyodbc://{username}:{password}@{server}/{database}"
    f"?driver={driver}&TrustServerCertificate=yes"
)

engine = create_engine(connection_string)



In [6]:
query = "SELECT * FROM dbo.presto_fake_data;"
df = pd.read_sql(query, engine)

df.head()


,Date,Year,Month,Weekday,Time,Hour,Transaction_Type,Amount,Location,Commute_Tag
0,2025-12-23,2025,December,Tuesday,15:10:00,15,Fare Payment,4.12,LESLIE ST / HIGHWAY 7,Work Commute
1,2025-12-01,2025,December,Monday,07:15:00,7,Fare Payment,4.12,HIGH TECH RD / RED MAPLE RD,Errand
2,2025-12-06,2025,December,Saturday,06:15:00,6,Free Transfer,0.00,YONGE ST / 16TH AVE,Work Commute
3,2025-10-16,2025,October,Thursday,07:30:00,7,Fare Payment,4.12,RICHMOND HILL CENTRE,Errand
4,2025-11-22,2025,November,Saturday,08:40:00,8,Fare Payment,4.12,DON MILLS STATION,Errand


In [7]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine

# --------- Paths (edit this root path once) ----------
PROJECT_ROOT = Path(r"C:\DataAnalysis\repos\project-01-transport-expenses")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --------- SQL connection ----------
server = "localhost,1433"
database = "TransportAnalytics"
username = "sa"
password = "StrongPass!123"
driver = "ODBC Driver 18 for SQL Server"

engine = create_engine(
    f"mssql+pyodbc://{username}:{password}@{server}/{database}"
    f"?driver={driver}&TrustServerCertificate=yes"
)

# --------- 1) Extract ----------
df = pd.read_sql("SELECT * FROM dbo.presto_fake_data;", engine)

# --------- 2) Clean / enforce types ----------
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# if Time comes in as object/string, force it
df["Time"] = pd.to_datetime(df["Time"].astype(str), format="%H:%M:%S", errors="coerce").dt.time

df["Hour"] = pd.to_numeric(df["Hour"], errors="coerce").astype("Int64")
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

# --------- 3) Feature engineering ----------
df["IsPaidFare"] = (df["Transaction_Type"] == "Fare Payment").astype(int)
df["MonthStart"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["WeekdayNum"] = df["Date"].dt.weekday  # 0=Mon

df["DateTime"] = pd.to_datetime(df["Date"].dt.date.astype(str) + " " + df["Hour"].astype(str).str.zfill(2) + ":00:00")

# --------- 4) Basic validation (cheap & important) ----------
bad_dates = df["Date"].isna().sum()
bad_time = df["Time"].isna().sum()
if bad_dates or bad_time:
    print(f"⚠️ Bad dates: {bad_dates}, bad times: {bad_time}")

# --------- 5) Save processed outputs ----------
# Best format for analytics
processed_parquet = PROCESSED_DIR / "presto_processed.parquet"
df.to_parquet(processed_parquet, index=False)

# Optional: CSV for easy viewing/Excel/Tableau
processed_csv = PROCESSED_DIR / "presto_processed.csv"
df.to_csv(processed_csv, index=False)

print("Saved:", processed_parquet)
print("Saved:", processed_csv)



InterfaceError: (pyodbc.InterfaceError) ('28000', '[28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Login failed for user \'sa\'. (18456) (SQLDriverConnect); [28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot open database "TransportAnalytics" requested by the login. The login failed. (4060); [28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Login failed for user \'sa\'. (18456); [28000] [Microsoft][ODBC Driver 18 for SQL Server][SQL Server]Cannot open database "TransportAnalytics" requested by the login. The login failed. (4060)')
(Background on this error at: https://sqlalche.me/e/20/rvf5)

In [8]:
import pyodbc
pyodbc.drivers()


['SQL Server',
 'Microsoft Access Driver (*.mdb, *.accdb)',
 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)',
 'Microsoft Access Text Driver (*.txt, *.csv)',
 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']